# Phase 11 Fine-Tuning — Preparation & Surgical Audit Notebook (Isolated)

This notebook is a step-by-step audit of the Phase 11 fine-tuning pipeline described in:
- `notebooks/phase11_finetune_replication.ipynb`

It is designed to be run **in strict isolation**:
- All generated artifacts go under a notebook-local workspace folder.
- The notebook avoids running any command known to write to `data/` or `tests/fixtures/` in the repo root.
- Heavy steps (LoRA training) are opt-in and gated by explicit flags.

If a step would write outside the notebook workspace, this notebook loads and analyses existing artifacts instead of regenerating them.


In [ ]:
"""Setup: strict isolation paths + safe subprocess runner."""
from __future__ import annotations

import json
import os
import subprocess
import sys
from dataclasses import dataclass
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path

NB_DIR = Path.cwd().resolve()
ROOT = NB_DIR.parent if (NB_DIR.parent / "src").exists() else NB_DIR

WORKSPACE = NB_DIR / "phase11_finetune_preparation_workspace"
WS_DATA = WORKSPACE / "data"
WS_LOGS = WORKSPACE / "logs"
WS_LEDGER = WORKSPACE / "reproduction_ledger.json"

WORKSPACE.mkdir(parents=True, exist_ok=True)
WS_DATA.mkdir(parents=True, exist_ok=True)
WS_LOGS.mkdir(parents=True, exist_ok=True)

ALLOW_NETWORK = False
RUN_RANK_SWEEP = False
RUN_FINETUNE = False

def _assert_within_workspace(path: Path) -> None:
    resolved = path.resolve()
    ws = WORKSPACE.resolve()
    if ws not in resolved.parents and resolved != ws:
        raise RuntimeError(f"Refusing to write outside workspace: {resolved}")

def _run(cmd: list[str], env: dict[str, str] | None = None) -> str:
    run_env = os.environ.copy()
    if env:
        run_env.update(env)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    log_path = WS_LOGS / f"{stamp}__{'_'.join(Path(c).name for c in cmd[:3])}.log"
    _assert_within_workspace(log_path)

    proc = subprocess.run(
        cmd,
        cwd=str(ROOT),
        env=run_env,
        text=True,
        capture_output=True,
    )
    out = (proc.stdout or "") + ("\n" + proc.stderr if proc.stderr else "")
    log_path.write_text(out)
    print(out)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit={proc.returncode}): {' '.join(cmd)}")
    return out

print(f"Repo root:   {ROOT}")
print(f"Notebook:    {NB_DIR}")
print(f"Workspace:   {WORKSPACE}")
print(f"WS data:     {WS_DATA}")
print(f"WS logs:     {WS_LOGS}")

---
## 0. Environment & dependency probe

This section verifies that the notebook kernel can run the Phase 11 preprocessing scripts (Parquet + JSONL generation) without installing anything.

If imports fail, install into *this notebook’s kernel environment* (do not run `make finetune-setup` from here).


In [ ]:
print(sys.version)
print(sys.executable)

deps = ["pandas", "pyarrow", "numpy", "matplotlib"]
missing = []
for name in deps:
    try:
        __import__(name)
    except Exception:
        missing.append(name)

if missing:
    print("Missing deps:", missing)
    print("Install into this kernel env (example):")
    print("  pip install " + " ".join(missing))
else:
    print("Core deps OK")

required_scripts = [
    ROOT / "scripts" / "acquire_phase1_data.py",
    ROOT / "scripts" / "acquire_phase10_data.py",
    ROOT / "scripts" / "generate_reference_strategies.py",
    ROOT / "scripts" / "generate_training_jsonl.py",
    ROOT / "scripts" / "generate_compliance_examples.py",
]
missing_scripts = [str(p) for p in required_scripts if not p.exists()]
if missing_scripts:
    raise FileNotFoundError("Missing scripts:\n" + "\n".join(missing_scripts))
print("Scripts OK")

---
## 1. Acquire OHLCV market data (isolated)

The Phase 11 label generation needs OHLCV Parquets for the full 15-ticker universe.

This notebook uses the underlying scripts directly and forces `--data-dir` to the notebook workspace.

Set `ALLOW_NETWORK = True` in the setup cell to enable downloads.


In [ ]:
market_dir = WS_DATA / "market"
market_dir.mkdir(parents=True, exist_ok=True)

if not ALLOW_NETWORK:
    print("Network disabled. Set ALLOW_NETWORK=True in the setup cell to download.")
else:
    _run([sys.executable, str(ROOT / "scripts" / "acquire_phase1_data.py"), "--data-dir", str(WS_DATA)])
    _run([sys.executable, str(ROOT / "scripts" / "acquire_phase10_data.py"), "--data-dir", str(WS_DATA)])

parquets = sorted(market_dir.glob("*.parquet"))
tickers = sorted({p.name.split("_")[0] for p in parquets})
print(f"Market parquet files: {len(parquets)}")
print(f"Tickers present: {tickers}")

---
## 2. Generate Dataset Family C reference strategies (isolated)

This runs `scripts/generate_reference_strategies.py` with workspace-local paths.

You can start with the 3-ticker pilot set (fast) and later expand to the 15-ticker universe.


In [ ]:
TICKERS = ["AAPL", "JPM", "XOM"]
HORIZON = 60

ref_out = WS_DATA / "reference_strategies"
ref_out.mkdir(parents=True, exist_ok=True)

tickers_arg = ",".join(TICKERS)

_run([
    sys.executable,
    str(ROOT / "scripts" / "generate_reference_strategies.py"),
    "--data-dir", str(WS_DATA),
    "--output-dir", str(ref_out),
    "--tickers", tickers_arg,
    "--horizon", str(HORIZON),
])

mr_dir = ref_out / "max_return"
ra_dir = ref_out / "risk_adjusted"

mr_files = sorted(mr_dir.glob(f"*_{HORIZON}d.parquet"))
ra_files = sorted(ra_dir.glob(f"*_{HORIZON}d.parquet"))

print(f"max_return files: {len(mr_files)}")
print(f"risk_adjusted files: {len(ra_files)}")
print("Sample:")
print("  ", mr_files[0] if mr_files else None)
print("  ", ra_files[0] if ra_files else None)

In [ ]:
import pandas as pd

if not mr_files:
    raise FileNotFoundError("No max_return Parquets found. Check Step 2 output.")

sample = mr_files[0]
df = pd.read_parquet(sample)
print(sample.name)
print(df.head(3))
print("Rows:", len(df))
if "label" in df.columns:
    print("Label counts:")
    print(df["label"].value_counts())

---
## 3. Generate Phase 11 JSONL training sets (isolated)

This runs `scripts/generate_training_jsonl.py` against the workspace reference strategies and writes JSONL files under `workspace/data/training/`.


In [ ]:
train_out = WS_DATA / "training"
train_out.mkdir(parents=True, exist_ok=True)

_run([
    sys.executable,
    str(ROOT / "scripts" / "generate_training_jsonl.py"),
    "--agent", "both",
    "--horizon", str(HORIZON),
    "--data-dir", str(WS_DATA),
    "--output-dir", str(train_out),
])

tech_jsonl = train_out / f"technical_max_return_{HORIZON}d.jsonl"
fund_jsonl = train_out / f"fundamental_risk_adjusted_{HORIZON}d.jsonl"

print("technical JSONL:", tech_jsonl, tech_jsonl.exists())
print("fundamental JSONL:", fund_jsonl, fund_jsonl.exists())
print("Sizes (bytes):")
for p in [tech_jsonl, fund_jsonl]:
    if p.exists():
        print(f"  {p.name}: {p.stat().st_size:,}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

if not tech_jsonl.exists():
    raise FileNotFoundError(f"Missing JSONL: {tech_jsonl}")

examples = []
with open(tech_jsonl) as fh:
    for i, line in enumerate(fh):
        if i >= 2000:
            break
        examples.append(json.loads(line))

print("Loaded examples (sample):", len(examples))
ex0 = examples[0]
for msg in ex0.get("messages", []):
    role = msg.get("role", "").upper()
    body = msg.get("content", "")
    trunc = body[:700] + " ...[truncated]" if len(body) > 700 else body
    print("\n[" + role + "]")
    print(trunc)

lengths = [sum(len(m.get("content", "")) for m in ex.get("messages", [])) // 4 for ex in examples]

plt.figure(figsize=(9, 3.5))
plt.hist(lengths, bins=50, color="steelblue", alpha=0.85, edgecolor="none")
plt.axvline(float(np.median(lengths)), color="firebrick", lw=1.3, ls="--")
plt.xlabel("Approx. tokens (chars/4)")
plt.ylabel("Count")
plt.title("Technical JSONL token-length distribution (sample)")
plt.tight_layout()
plt.show()

print("Token stats (sample):")
print("  min:", int(np.min(lengths)))
print("  median:", float(np.median(lengths)))
print("  max:", int(np.max(lengths)))

---
## 4. Compliance examples (writes only inside workspace)

This script reads existing verified baseline fixtures under `tests/fixtures/` (read-only) and writes JSONL files into the workspace.


In [ ]:
_run([
    sys.executable,
    str(ROOT / "scripts" / "generate_compliance_examples.py"),
    "--output-dir", str(train_out),
])

out_files = sorted(train_out.glob("*_compliance*.jsonl"))
for p in out_files:
    n_lines = sum(1 for _ in open(p))
    print(f"{p.name}: {n_lines} lines")

---
## 5. Optional: LoRA rank sweep + fine-tuning (heavy, opt-in)

This section is disabled by default.

Constraints:
- The fine-tune script uses `venvs/finetune/bin/python` from the repo root.
- Training requires a local MLX model at `~/.lmstudio/models/...`.

If prerequisites are missing, the notebook refuses to run training.


In [ ]:
finetune_python = ROOT / "venvs" / "finetune" / "bin" / "python"
model_base = Path.home() / ".lmstudio" / "models" / "lmstudio-community" / "Qwen2.5-Coder-32B-Instruct-MLX-8bit"

print("finetune venv python exists:", finetune_python.exists(), finetune_python)
print("base model path exists:", model_base.exists(), model_base)

if RUN_RANK_SWEEP:
    if not finetune_python.exists() or not model_base.exists():
        raise RuntimeError("Missing prerequisites for rank sweep. See prints above.")
    _run([
        sys.executable,
        str(ROOT / "scripts" / "run_phase11_finetune.py"),
        "--rank-sweep",
        "--sweep-iters", "5",
        "--agent", "technical",
        "--data-dir", str(WS_DATA),
    ])

if RUN_FINETUNE:
    if not finetune_python.exists() or not model_base.exists():
        raise RuntimeError("Missing prerequisites for fine-tune. See prints above.")
    _run([
        sys.executable,
        str(ROOT / "scripts" / "run_phase11_finetune.py"),
        "--iters", "10",
        "--agent", "both",
        "--data-dir", str(WS_DATA),
    ])

sweep_path = train_out / "rank_sweep_results.json"
if sweep_path.exists():
    raw = json.loads(sweep_path.read_text())
    print("rank sweep results keys:", sorted(raw.keys()))
else:
    print("No rank sweep results in workspace.")

---
## 6. Evaluation fixture analysis (read-only)

The script `scripts/run_phase11_evaluation.py` writes to `tests/fixtures/baseline/phase11_evaluation.json` in the repo root.

To preserve strict isolation, this notebook does not execute that script. Instead it loads the existing fixture and reproduces the core plots.


In [ ]:
try:
    from IPython.display import display as _display
except Exception:
    def _display(x):
        print(x)

fixture = ROOT / "tests" / "fixtures" / "baseline" / "phase11_evaluation.json"
if not fixture.exists():
    raise FileNotFoundError(f"Missing fixture: {fixture}")

payload = json.loads(fixture.read_text())
rows = payload.get("results", [])
df = pd.DataFrame(rows)
print("Rows:", len(df))
keep = [
    "ticker",
    "base_technical_gr",
    "finetuned_technical_gr",
    "base_fundamental_gr",
    "finetuned_fundamental_gr",
    "base_pairwise_diversity",
    "finetuned_pairwise_diversity",
]
cols = [c for c in keep if c in df.columns]
_display(df[cols])

if "base_pairwise_diversity" in df.columns and "finetuned_pairwise_diversity" in df.columns:
    plt.figure(figsize=(8, 3.5))
    x = np.arange(len(df))
    plt.bar(x - 0.18, df["base_pairwise_diversity"], width=0.36, label="base", color="steelblue", alpha=0.85)
    plt.bar(x + 0.18, df["finetuned_pairwise_diversity"], width=0.36, label="finetuned", color="seagreen", alpha=0.85)
    plt.xticks(x, df["ticker"].tolist())
    plt.ylim(0, 1)
    plt.title("Phase 11 fixture: diversity preserved")
    plt.legend()
    plt.tight_layout()
    plt.show()

---
## 7. Workspace reproduction ledger (hashes)

This writes a provenance ledger **inside the workspace**: file list + SHA-256 hashes.


In [ ]:
@dataclass(frozen=True)
class LedgerEntry:
    relpath: str
    sha256: str
    bytes: int

def _hash_file(p: Path) -> LedgerEntry:
    h = sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return LedgerEntry(relpath=str(p.relative_to(WORKSPACE)), sha256=h.hexdigest(), bytes=p.stat().st_size)

files = [p for p in WORKSPACE.rglob("*") if p.is_file() and p.name != WS_LEDGER.name]
entries = [_hash_file(p) for p in sorted(files)]

ledger = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "workspace": str(WORKSPACE),
    "entries": [e.__dict__ for e in entries],
}

WS_LEDGER.write_text(json.dumps(ledger, indent=2))
print("Ledger written:", WS_LEDGER)
print("Files hashed:", len(entries))